### Read in user condition assignments

In [2]:
generated_users = {}
with open("generated_combined/user_log_generated_combined.txt", "r") as infile:
    for line in infile:
        if "GROUP" in line:
            user, group = line.split("||")
            user = user.split(":")[1].strip()
            group = group.split(":")[1].strip()
            generated_users[user] = group

real_users = {}
with open("cts_combined/combined_user_log.txt", "r") as infile:
    for line in infile:
        if "GROUP" in line:
            user, group = line.split("||")
            user = user.split(":")[1].strip()
            group = group.split(":")[1].strip()
            real_users[user] = group


### Parse Surveys

In [3]:
import ast
post_surveys = []
pre_surveys = []
post_survey_users = []
with open("cts_combined/combined_survey_log.txt", "r") as infile:
    for line in infile:
        user, survey = line.split("||")
        user = user.split(":")[1].strip()
        survey = survey[13:].strip()
        survey = ast.literal_eval(survey)
        survey["user"] = user
        if "POST-SURVEY" in line:
            post_survey_users.append(user)
            post_surveys.append(survey)
        else:
            pre_surveys.append(survey)

with open("generated_combined/survey_log_generated_combined.txt", "r") as infile:
    for line in infile:
        user, survey = line.split("||")
        user = user.split(":")[1].strip()
        survey = survey[13:].strip()
        survey = ast.literal_eval(survey)
        survey["user"] = user
        if "POST-SURVEY" in line:
            post_survey_users.append(user)
            post_surveys.append(survey)
        else:
            pre_surveys.append(survey)

# filter out anyone who didn't complete a post-survey
real_users = {u: real_users[u] for u in real_users if u in post_survey_users}
generated_users = {u: generated_users[u] for u in generated_users if u in post_survey_users}



### Understand Previous User Experience with Business Travel

In [7]:
# Figure out distribution of previous business travel experience
avg_real_usr_exp = []
avg_generated_exp = []

for survey in pre_surveys:
    user = survey["user"]
    if user in real_users:
        avg_real_usr_exp.append(int(survey["experience_businesstravel"]))
    elif user in generated_users:
        avg_generated_exp.append(int(survey["experience_businesstravel"]))

print(avg_generated_exp)
print(avg_real_usr_exp)
print(sum(avg_generated_exp)/len(avg_generated_exp))
print(sum(avg_real_usr_exp)/len(avg_real_usr_exp))
print(min(avg_generated_exp), max(avg_generated_exp))
print(min(avg_real_usr_exp), max(avg_real_usr_exp))

[2, 3, 2, 3, 3, 2, 1, 1, 1, 3, 3, 1, 3, 1, 2, 4, 4, 3, 2]
[1, 3, 3, 1, 1, 2, 3, 3, 1, 1, 1, 2, 3, 1, 1, 2, 3, 3, 1, 2, 2, 3]
2.3157894736842106
1.9545454545454546
1 4
1 3


### Get Real Objective Metrics From Filtered Transcript

In [19]:
# chat_stats = {"real": {}, "generated": {}}
goal_type = None
dialogs = []

user_dialog_nums = {}
user_end_conditions = {}

with open("generated_combined/transcript_generated_combined.txt", "r") as transcript:
    for line in transcript:
        # get meta data about dialog
        if "GOAL-TYPE" in line:
            current_dialog = {}
            tmp = line.split()
            user = tmp[1].strip()
            policy = tmp[3].strip(")").strip()
            # current_dialog["policy"] = policy
            policy = "generated"
            current_dialog['user'] = user
            current_dialog['turns'] = []
            goal_type = tmp[-1].strip(")").strip()
            current_dialog["goal_type"] = goal_type
        # only include actual user utterances, not processing steps
        elif "USER:" in line and not "POST-NLU" in line:
            current_dialog["turns"].append(line)
        elif "SYSTEM" in line:
            current_dialog['turns'].append(line)
        elif "DIALOG END:" in line:
            current_dialog["end_condition"] = line.split(":")[1].strip()
        elif "SUBJECTIVE LENGTH" in line:
            current_dialog["sub_length"] = line.split(":")[1].strip()
        elif "SUBJECTIVE QUALITY" in line:
            current_dialog["sub_quality"] = line.split(":")[1].strip()
        # dialogs are separated by a newline in the processed transcripts, so look for that here
        elif line.strip() == "":
            if current_dialog:
                # update for length of all turns (not just system utterances)
                obj_length = len(current_dialog["turns"])

                # initialize dict
                if not goal_type in chat_stats[policy]:
                    chat_stats[policy][goal_type] = {"length": [], "end_condition": [], "sub_length": [], "sub_quality": []}

                # collect statistics for this conversation only if the user also filled out a post-survey
                if current_dialog["user"] in generated_users and current_dialog['user'] != "57affbcf1a53cf8152a4f84b337572":
                    # chat_stats[policy][goal_type]["length"].append(obj_length)
                    # chat_stats[policy][goal_type]["end_condition"].append(current_dialog["end_condition"])
                    # chat_stats[policy][goal_type]["sub_length"].append(int(current_dialog["sub_length"]))
                    # chat_stats[policy][goal_type]["sub_quality"].append(int(current_dialog["sub_quality"]))
                    if current_dialog['user'] not in user_dialog_nums:
                        user_dialog_nums[current_dialog['user']] = 0
                    user_dialog_nums[current_dialog['user']] += 1

                    if current_dialog["user"] not in user_end_conditions:
                        user_end_conditions[current_dialog["user"]] = []
                    user_end_conditions[current_dialog["user"]].append( 1 if current_dialog["end_condition"] == "SUCCESS" else 0)

                    dialogs.append(current_dialog)
                else:
                    print(user)
                current_dialog = {}

b32326eb99667cb76c264f0af58386
b32326eb99667cb76c264f0af58386
b32326eb99667cb76c264f0af58386


### Remove user's who didn't interact with system

In [131]:
user_black_list = set()
user_black_list.add("57affbcf1a53cf8152a4f84b337572")
user_gray_list = set()

for user in users:
    if user in user_dialog_nums:
        if user_dialog_nums[user] != 3:
            user_gray_list.add(user)
    else:
        user_black_list.add(user)

print("Removed: ", user_black_list)
print("To Investigate: ", user_gray_list)
users = {u: users[u] for u in users if u not in user_black_list}

Removed:  {'57affbcf1a53cf8152a4f84b337572'}
To Investigate:  {'85452be9505ef3ee3499d59b5300f3'}


In [20]:
l = {}
ec = {}
sl = {}
sq = {}

for policy_type in chat_stats:
    print(f"POLICY: {policy_type}")
    l[policy_type] = []
    ec[policy_type] = []
    sl[policy_type] = []
    sq[policy_type] = []
    for goal_type in chat_stats[policy_type]:
        lengths = chat_stats[policy_type][goal_type]["length"]
        end_conditions = chat_stats[policy_type][goal_type]["end_condition"]
        sub_lengths = chat_stats[policy_type][goal_type]["sub_length"]
        sub_qualities = chat_stats[policy_type][goal_type]["sub_quality"]

        avg_length = sum(lengths)/len(lengths)
        avg_success = end_conditions.count("SUCCESS")/len(end_conditions)
        avg_sub_length = sum(sub_lengths)/len(sub_lengths)
        avg_sub_quality = sum(sub_qualities)/len(sub_qualities)

        l[policy_type] += lengths
        ec[policy_type] += end_conditions
        sl[policy_type] += sub_lengths
        sq[policy_type] += sub_qualities

        print(f"{goal_type} -- AVG. LEN: {avg_length}  AVG. SUCCESS: {avg_success}  AVG. SUBJECTIVE LENGTH: {avg_sub_length}  AVG. SUBJECTIVE QUALITY: {avg_sub_quality}")


    print(f"COMBINED -- AVG. LEN: {sum(l[policy_type])/len(l[policy_type])}  AVG. SUCCESS: {ec[policy_type].count('SUCCESS')/len(ec[policy_type])}  AVG. SUBJECTIVE LENGTH: {sum(sl[policy_type])/len(sl[policy_type])}  AVG. SUBJECTIVE QUALITY: {sum(sq[policy_type])/len(sq[policy_type])}")

POLICY: real
OPEN -- AVG. LEN: 7.421052631578948  AVG. SUCCESS: 0.9473684210526315  AVG. SUBJECTIVE LENGTH: 3.0  AVG. SUBJECTIVE QUALITY: 2.8947368421052633
EASY -- AVG. LEN: 3.8  AVG. SUCCESS: 0.8  AVG. SUBJECTIVE LENGTH: 2.6  AVG. SUBJECTIVE QUALITY: 3.0
HARD -- AVG. LEN: 7.388888888888889  AVG. SUCCESS: 0.5555555555555556  AVG. SUBJECTIVE LENGTH: 3.0  AVG. SUBJECTIVE QUALITY: 2.9444444444444446
COMBINED -- AVG. LEN: 6.140350877192983  AVG. SUCCESS: 0.7719298245614035  AVG. SUBJECTIVE LENGTH: 2.8596491228070176  AVG. SUBJECTIVE QUALITY: 2.9473684210526314
POLICY: generated
OPEN -- AVG. LEN: 8.210526315789474  AVG. SUCCESS: 0.8421052631578947  AVG. SUBJECTIVE LENGTH: 2.736842105263158  AVG. SUBJECTIVE QUALITY: 2.8421052631578947
EASY -- AVG. LEN: 2.1666666666666665  AVG. SUCCESS: 0.8888888888888888  AVG. SUBJECTIVE LENGTH: 2.4444444444444446  AVG. SUBJECTIVE QUALITY: 2.9444444444444446
HARD -- AVG. LEN: 5.277777777777778  AVG. SUCCESS: 0.4444444444444444  AVG. SUBJECTIVE LENGTH: 2.777

### CTS Chat statistics

| Goal Type | Avg. Dialog Length| Avg. Success | Avg. Subjective Length | Avg. Subjective Quality |
|-----------|-------------------|--------------|------------------------|-------------------------|
| OPEN      | 9.30              | 0.90         | 3.05                   | 2.80                    |
| EASY      | 4.76              | 0.76         | 2.67                   | 2.90                    |
| HARD      | 8.20              | 0.55         | 3.05                   | 2.90                    |
| COMBINED  | 7.38              | 0.74         | 2.92                   | 2.87                    |

### FAQ Chat statistics

| Goal Type | Avg. Dialog Length| Avg. Success | Avg. Subjective Length | Avg. Subjective Quality |
|-----------|-------------------|--------------|------------------------|-------------------------|
| OPEN      | 2.20              | 0.65         | 2.20                   | 2.60                    |
| EASY      | 2.19              | 0.81         | 2.23                   | 2.71                    |
| HARD      | 2.40              | 0.25         | 2.35                   | 2.50                    |
| COMBINED  | 2.26              | 0.57         | 2.28                   | 2.60                    |

### HDC Chat statistics

| Goal Type | Avg. Dialog Length| Avg. Success | Avg. Subjective Length | Avg. Subjective Quality |
|-----------|-------------------|--------------|------------------------|-------------------------|
| OPEN      | 16.50             | 0.77         | 3.23                   | 2.91                    |
| EASY      | 10.59             | 0.32         | 3.18                   | 2.50                    |
| HARD      | 12.86             | 0.23         | 2.82                   | 1.82                    |
| COMBINED  | 13.32             | 0.44         | 3.08                   | 2.41                    |

### Perform Statistical Tests for Objective Measures

In [22]:
from scipy import stats

print("LENGTH")
print(stats.ttest_ind(l["real"], l["generated"]))

LENGTH
Ttest_indResult(statistic=0.849237114925752, pvalue=0.39759381118521286)


**Takeaway:**
 * CTS is statistically significantly faster than HDC

In [37]:
print("SUBJECTIVE LENGTH")
print(stats.ttest_ind(sl["real"], sl["generated"]))
print(np.mean(sl["real"]), np.mean(sl["generated"]))

SUBJECTIVE LENGTH
Ttest_indResult(statistic=1.430313297656477, pvalue=0.15546155129714695)
2.8596491228070176 2.6545454545454548


In [31]:
print(stats.ttest_ind([1 if el == "SUCCESS" else 0 for el in ec["real"]], [1 if el == "SUCCESS" else 0 for el in ec["generated"]]))

SUCCESS
Ttest_indResult(statistic=0.5414581907049234, pvalue=0.58928679200614)


In [38]:
print("SUBJECTIVE QUALITY")
print(stats.ttest_ind(sq["real"], sq["generated"]))
print(np.mean(sq["real"]), np.mean(sq["generated"]))

SUBJECTIVE QUALITY
Ttest_indResult(statistic=1.0978498494331959, pvalue=0.27466739431498166)
2.9473684210526314 2.727272727272727


### Compare to simulator

In [36]:
real_simulator = [0] * 130 + [1] * 370
generated_simulator = [0] * 152 + [1] * 348

# Test to see if the variance is within 4 times for the larger group, if so, use Welch's T-test, if not, the normal one is fine
r_success = [1 if el == "SUCCESS" else 0 for el in ec["real"]]
g_success = [1 if el == "SUCCESS" else 0 for el in ec["generated"]]
print(np.var(real_simulator), np.var(r_success))
print(np.var(generated_simulator), np.var(g_success))

# can use normal t-test because variance in basically the same
print(stats.ttest_ind(real_simulator, r_success))
print(stats.ttest_ind(generated_simulator, g_success))

0.19240000000000004 0.1760541705140043
0.21158399999999997 0.19834710743801648
Ttest_indResult(statistic=-0.5220401020255736, pvalue=0.6018506933242547)
Ttest_indResult(statistic=-0.47919328160339436, pvalue=0.6319905667408593)


### Save survey data to csv format

In [134]:
# Filter out users who didn't complete a post-survey/dialog or didn't try to interact with the system
pre_surveys = [survey for survey in pre_surveys if survey["user"] in users]
post_surveys = [survey for survey in post_surveys if survey["user"] in users]

import csv
with open("hdc_faq_combined/pre_survey.csv", "w", newline='') as outfile:
    fieldnames = pre_surveys[0].keys()
    writer = csv.DictWriter(outfile, fieldnames=fieldnames, delimiter="|")
    writer.writeheader()
    for s in pre_surveys:
        if s['user'] in user_black_list:
            continue
        writer.writerow(s)

with open("hdc_faq_combined/post_survey.csv", "w", newline='') as outfile:
    fieldnames = post_surveys[0].keys()
    writer = csv.DictWriter(outfile, fieldnames=fieldnames, delimiter="|")
    writer.writeheader()
    for s in post_surveys:
        if s['user'] in user_black_list:
            continue
        writer.writerow(s)

### Parse out Trust and Usability scores

In [113]:
import numpy as np

trust = {"cts": [], "faq": [], "hdc": []}
reliability = {"cts": [], "faq": [], "hdc": []}
usability = {"cts": [], "faq": [], "hdc": []}

u_usability = {}
u_trust = {}
u_reliability = {}

for res in post_surveys:
    user = res["user"]
    if user in users:
        policy = users[user]
        user_reliability = (int(res["reliability_1"]) + int(res["reliability_2"]) + (6 - int(res["reliability_3"])) + int(res["reliability_4"]) + (6 - int(res["reliability_5"])) + int(res["reliability_6"])) / 6
        user_trust = (int(res["trust_1"]) + int(res["trust_2"])) / 2
        # should be 0 to 4 scale, not 1 to 5
        user_usability = ((int(res['umux_1']) - 1) + (5 - int(res['umux_2'])) + (int(res['umux_3']) - 1) + (5 - int(res['umux_4']))) / 16 * 100
        u_usability[user] = user_usability
        u_trust[user] = user_trust
        u_reliability[user] = user_reliability

        usability[policy].append(user_usability)
        trust[policy].append(user_trust)
        reliability[policy].append(user_reliability)

In [127]:
print('USABILITY')
print(np.mean(usability["cts"]), np.mean(usability["faq"]), np.mean(usability["hdc"]))
print(stats.f_oneway(usability["cts"], usability["faq"], usability["hdc"]))
print(stats.tukey_hsd(usability["cts"], usability["faq"], usability["hdc"]))

print('TRUST')
print(np.mean(trust["cts"]), np.mean(trust["faq"]), np.mean(trust["hdc"]))
print(stats.f_oneway(trust["cts"], trust["faq"], trust["hdc"]))
print(stats.tukey_hsd(trust["cts"], trust["faq"], trust["hdc"]))

print('RELIABILITY')
print(np.mean(reliability["cts"]), np.mean(reliability["faq"]), np.mean(reliability["hdc"]))
print(stats.f_oneway(reliability["cts"], reliability["faq"], reliability["hdc"]))
print(stats.tukey_hsd(reliability["cts"], reliability["faq"], reliability["hdc"]))

USABILITY
60.0 57.73809523809524 36.93181818181818
F_onewayResult(statistic=5.514770440087835, pvalue=0.006329469138969568)
Tukey's HSD Pairwise Group Comparisons (95.0% Confidence Interval)
Comparison  Statistic  p-value  Lower CI  Upper CI
 (0 - 1)      2.262     0.955   -16.553    21.077
 (0 - 2)     23.068     0.011     4.463    41.673
 (1 - 0)     -2.262     0.955   -21.077    16.553
 (1 - 2)     20.806     0.023     2.435    39.178
 (2 - 0)    -23.068     0.011   -41.673    -4.463
 (2 - 1)    -20.806     0.023   -39.178    -2.435

TRUST
3.05 2.8333333333333335 2.6136363636363638
F_onewayResult(statistic=1.00538922481986, pvalue=0.3719820754054256)
Tukey's HSD Pairwise Group Comparisons (95.0% Confidence Interval)
Comparison  Statistic  p-value  Lower CI  Upper CI
 (0 - 1)      0.217     0.767    -0.532     0.965
 (0 - 2)      0.436     0.339    -0.304     1.176
 (1 - 0)     -0.217     0.767    -0.965     0.532
 (1 - 2)      0.220     0.751    -0.511     0.950
 (2 - 0)     -0.436 

**Takeaways:**
 * Much more usable than HDC
 * Trust and reliability maybe require more domain knowledge than the users had in order to decide if an answer seemed correct to them or not

### What mental models did users have?

In [121]:
mental_models = { 
    "cts": {
      "natural language": [],
      "keywords": [],
      "specific question": [],
      "follow-up questions": [],
      "general answer": [],
      "personalized answer": [],
      "immediate answer": [],
      "long dialog": []
      },
    "faq": {
      "natural language": [],
      "keywords": [],
      "specific question": [],
      "follow-up questions": [],
      "general answer": [],
      "personalized answer": [],
      "immediate answer": [],
      "long dialog": []
    },
    "hdc": {
      "natural language": [],
      "keywords": [],
      "specific question": [],
      "follow-up questions": [],
      "general answer": [],
      "personalized answer": [],
      "immediate answer": [],
      "long dialog": []
    }
}

gender = {"cts": [], "faq": [], "hdc": []}
age = {"cts": [], "faq": [], "hdc": []}
chatbot_xp = {"cts": [], "faq": [], "hdc": []}
travel_xp = {"cts": [], "faq": [], "hdc": []}

user_mms = {}

for res in pre_surveys:
    user = res["user"]
    if user in users:
      policy = users[user]
      mental_models[policy]["natural language"].append(int(res["chat_exp_1"]))
      mental_models[policy]["keywords"].append(int(res["chat_exp_2"]))
      mental_models[policy]["specific question"].append(int(res["chat_exp_3"]))
      mental_models[policy]["follow-up questions"].append(int(res['chat_exp_4']))
      mental_models[policy]["general answer"].append(int(res["chat_exp_5"]))
      mental_models[policy]["personalized answer"].append(int(res["chat_exp_6"]))
      mental_models[policy]["immediate answer"].append(int(res['chat_exp_7']))
      mental_models[policy]["long dialog"].append(int(res["chat_exp_8"]))
      user_mms[user] = {key: mental_models[policy][key][-1] for key in mental_models[policy]}

      gender[policy].append(res["gender"])
      age[policy].append(res["age"])
      chatbot_xp[policy].append(int(res["experience_chatbots"]))
      travel_xp[policy].append(int(res["experience_businesstravel"]))

## Demographic Information

In [124]:

for policy in ["cts", "faq", "hdc"]:
    print(policy)
    print(f"GENDER BREAKDOWN: {gender[policy].count('male')} men, {gender[policy].count('female')} women, {gender[policy].count('other')} other")
    print(f"PREVIOUS CHATBOT XP: {sum(chatbot_xp[policy])/len(chatbot_xp[policy])}")
    print(f"PREVIOUS BUSINESS TRAVEL XP: {sum(travel_xp[policy])/len(travel_xp[policy])}")
    age_distro = {}
    for a in age[policy]:
        if a not in age_distro:
            age_distro[a] = 0
        age_distro[a] += 1

    print("AGE BREAKDOWN", age_distro)

print(stats.f_oneway(travel_xp["cts"], travel_xp["faq"], travel_xp["hdc"]))

cts
GENDER BREAKDOWN: 6 men, 13 women, 1 other
PREVIOUS CHATBOT XP: 2.9
PREVIOUS BUSINESS TRAVEL XP: 2.0
AGE BREAKDOWN {'30-39': 7, '40-49': 5, '50-59': 3, '20-29': 5}
faq
GENDER BREAKDOWN: 6 men, 15 women, 0 other
PREVIOUS CHATBOT XP: 3.0952380952380953
PREVIOUS BUSINESS TRAVEL XP: 1.9523809523809523
AGE BREAKDOWN {'30-39': 10, '40-49': 4, '20-29': 6, '50-59': 1}
hdc
GENDER BREAKDOWN: 8 men, 14 women, 0 other
PREVIOUS CHATBOT XP: 3.0
PREVIOUS BUSINESS TRAVEL XP: 1.7272727272727273
AGE BREAKDOWN {'30-39': 9, '20-29': 9, '60-69': 2, '40-49': 2}
F_onewayResult(statistic=0.5099058135758361, pvalue=0.6031309752245437)


### What role does Mental model have on dialog length?

In [46]:
for key in mental_models:
    yes = []
    no = []
    for d in dialogs:
        user = d["user"]
        if user in user_black_list:
            continue
        length = d["length"]
        if user_mms[user][key] >= 3:
            yes.append(length)
        else:
            no.append(length)
    res = stats.ttest_ind(yes, no)
    print(f" {key}: {res}")
    print(np.mean(yes), np.mean(no))

 natural language: Ttest_indResult(statistic=1.9155266761964809, pvalue=0.060359752761035515)
8.568181818181818 4.1875
 keywords: Ttest_indResult(statistic=-1.4117228115590934, pvalue=0.16337411360003143)
6.4523809523809526 9.61111111111111
 specific question: Ttest_indResult(statistic=-7.287401086047115, pvalue=9.639731742090734e-10)
6.140350877192983 31.333333333333332
 follow-up questions: Ttest_indResult(statistic=1.566282149091808, pvalue=0.12272278505647201)
8.282608695652174 4.5
 general answer: Ttest_indResult(statistic=-2.0515933337032517, pvalue=0.04473377705644093)
6.0476190476190474 10.555555555555555
 personalized answer: Ttest_indResult(statistic=1.9156184160699379, pvalue=0.060347821357475386)
9.615384615384615 5.705882352941177
 immediate answer: Ttest_indResult(statistic=-2.737908030429897, pvalue=0.008197087278619494)
6.346153846153846 14.25
 long dialog: Ttest_indResult(statistic=-1.3108316230119241, pvalue=0.19508128553310636)
6.622222222222222 9.733333333333333


### What role do expectations have on success?

In [37]:
for key in mental_models:
    yes = []
    no = []
    for d in dialogs:
        user = d["user"]
        if user in user_black_list:
            continue
        success = 1 if d["end_condition"] == "SUCCESS" else 0
        if user_mms[user][key] >= 3:
            yes.append(success)
        else:
            no.append(success)
    res = stats.ttest_ind(yes, no)
    print(f" {key}: {res}")

 natural language: Ttest_indResult(statistic=-0.8269903956433609, pvalue=0.4116319732672761)
 keywords: Ttest_indResult(statistic=0.1252872207611868, pvalue=0.9007294276197435)
 specific question: Ttest_indResult(statistic=3.1329514224425212, pvalue=0.002712743554706298)
 follow-up questions: Ttest_indResult(statistic=0.1810203347193921, pvalue=0.8569827002927071)
 general answer: Ttest_indResult(statistic=0.1252872207611868, pvalue=0.9007294276197435)
 personalized answer: Ttest_indResult(statistic=-1.8266946886631032, pvalue=0.07289307363436111)
 immediate answer: Ttest_indResult(statistic=1.6110350058128282, pvalue=0.11260141903011442)
 long dialog: Ttest_indResult(statistic=0.6653931857203331, pvalue=0.5084365638575771)


Mental models have no impact on actual success in the cts setting

### Role of Mental Models on Usability

In [50]:
for key in mental_models:
    yes = []
    no = []
    for d in dialogs:
        user = d["user"]
        if user not in u_usability:
            continue
        usability = u_usability[user]
        if user_mms[user][key] >= 3:
            yes.append(usability)
        else:
            no.append(usability)
    res = stats.ttest_ind(yes, no)
    print(f" {key}: {res}")
    print(np.mean(yes), np.mean(no))

 natural language: Ttest_indResult(statistic=3.3885714399744162, pvalue=0.0013467085431713866)
70.0657894736842 48.828125
 keywords: Ttest_indResult(statistic=-2.5762310073174417, pvalue=0.0128680901981009)
59.67261904761905 78.125
 specific question: Ttest_indResult(statistic=nan, pvalue=nan)
63.773148148148145 nan
 follow-up questions: Ttest_indResult(statistic=-0.5991009844236858, pvalue=0.551706834389919)
62.65625 66.96428571428571
 general answer: Ttest_indResult(statistic=-1.1186421318804998, pvalue=0.26843395279352084)
61.904761904761905 70.3125
 personalized answer: Ttest_indResult(statistic=-0.27298094776173853, pvalue=0.7859490761137501)
62.77173913043478 64.51612903225806
 immediate answer: Ttest_indResult(statistic=1.96568974675577, pvalue=0.05468496528579534)
65.68877551020408 45.0
 long dialog: Ttest_indResult(statistic=-3.3807546028242754, pvalue=0.0013784380994951535)
59.44444444444444 85.41666666666667


/Users/vanderly/Documents/conversational-tree-search/.env/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3464: RuntimeWarning:

Mean of empty slice.

/Users/vanderly/Documents/conversational-tree-search/.env/lib/python3.9/site-packages/numpy/core/_methods.py:192: RuntimeWarning:

invalid value encountered in scalar divide



Natural Language/Keyword expectation and dialog length expectation had a significant effect on usability

### Role of Mental Models on Reliability

In [47]:
for key in mental_models:
    if key == "specific question":
        continue
    yes = []
    no = []
    for d in dialogs:
        user = d["user"]
        if user not in u_reliability:
            continue
        reliability = u_reliability[user]
        if user_mms[user][key] >= 3:
            yes.append(reliability)
        else:
            no.append(reliability)
    res = stats.ttest_ind(yes, no)
    print(f" {key}: {res}")
    print(np.mean(yes), np.mean(no))

 natural language: Ttest_indResult(statistic=2.8722714058793968, pvalue=0.005885456995905381)
3.1447368421052633 2.5625
 keywords: Ttest_indResult(statistic=-1.5231136148197686, pvalue=0.13378910062742932)
2.892857142857143 3.25
 follow-up questions: Ttest_indResult(statistic=0.04712635655906155, pvalue=0.9625930514377123)
2.975 2.9642857142857144
 general answer: Ttest_indResult(statistic=0.2983017888301238, pvalue=0.7666615844602919)
2.988095238095238 2.9166666666666665
 personalized answer: Ttest_indResult(statistic=-0.135757599226155, pvalue=0.8925370991661421)
2.9565217391304346 2.9838709677419355
 immediate answer: Ttest_indResult(statistic=1.5485092932887359, pvalue=0.12756491214492688)
3.020408163265306 2.5
 long dialog: Ttest_indResult(statistic=-3.806681716150352, pvalue=0.0003729921372636551)
2.8222222222222224 3.7222222222222223


/var/folders/xn/1drxn9fx2dxc6_bh03c675ym0000gq/T/ipykernel_43639/3629777233.py:15: RuntimeWarning:

Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.



Natural language and dialog length had a significant effect on pereceived reliability

### Effect of mental models on trust

In [51]:
for key in mental_models:
    yes = []
    no = []
    for d in dialogs:
        user = d["user"]
        if user not in u_trust:
            continue
        trust = u_trust[user]
        if user_mms[user][key] >= 3:
            yes.append(trust)
        else:
            no.append(trust)
    res = stats.ttest_ind(yes, no)
    print(f" {key}: {res}")
    print(np.mean(yes), np.mean(no))

 natural language: Ttest_indResult(statistic=3.0978078171782264, pvalue=0.0031411003358131915)
3.3947368421052633 2.6875
 keywords: Ttest_indResult(statistic=-1.515521608753705, pvalue=0.13569604804111274)
3.0952380952380953 3.5
 specific question: Ttest_indResult(statistic=nan, pvalue=nan)
3.185185185185185 nan
 follow-up questions: Ttest_indResult(statistic=-0.5255714710481171, pvalue=0.6014201184619565)
3.15 3.2857142857142856
 general answer: Ttest_indResult(statistic=-1.515521608753705, pvalue=0.13569604804111274)
3.0952380952380953 3.5
 personalized answer: Ttest_indResult(statistic=-1.4333733628808447, pvalue=0.15773674450371405)
3.0 3.3225806451612905
 immediate answer: Ttest_indResult(statistic=1.6923001635724317, pvalue=0.0965732719810546)
3.2448979591836733 2.6
 long dialog: Ttest_indResult(statistic=-3.587070624426243, pvalue=0.0007387871047036022)
3.022222222222222 4.0


/Users/vanderly/Documents/conversational-tree-search/.env/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3464: RuntimeWarning:

Mean of empty slice.

/Users/vanderly/Documents/conversational-tree-search/.env/lib/python3.9/site-packages/numpy/core/_methods.py:192: RuntimeWarning:

invalid value encountered in scalar divide

/var/folders/xn/1drxn9fx2dxc6_bh03c675ym0000gq/T/ipykernel_43639/195690384.py:13: RuntimeWarning:

Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.



Natural langauge and expected dialog length had a significant effect on trust

### Get Post-Interaction Mental Models

In [91]:
post_mental_models = {
    "cts": {
      "natural language": [],
      "keywords": [],
      "specific question": [],
      "follow-up questions": [],
      "general answer": [],
      "personalized answer": [],
      "immediate answer": [],
      "long dialog": []
    },
    "faq": {
      "natural language": [],
      "keywords": [],
      "specific question": [],
      "follow-up questions": [],
      "general answer": [],
      "personalized answer": [],
      "immediate answer": [],
      "long dialog": []
    },
    "hdc": {
      "natural language": [],
      "keywords": [],
      "specific question": [],
      "follow-up questions": [],
      "general answer": [],
      "personalized answer": [],
      "immediate answer": [],
      "long dialog": []
    }
  }

post_user_mms = {}

for res in post_surveys:
    user = res["user"]
    if user not in users:
        continue
    policy = users[user]
    if res["chat_exp_1"] != "None":
        post_mental_models[policy]["natural language"].append(int(res["chat_exp_1"]))
    if res["chat_exp_2"] != "None":
        post_mental_models[policy]["keywords"].append(int(res["chat_exp_2"]))
    if res["chat_exp_3"] != "None":
        post_mental_models[policy]["specific question"].append(int(res["chat_exp_3"]))
    if res["chat_exp_4"] != "None":
        post_mental_models[policy]["follow-up questions"].append(int(res['chat_exp_4']))
    if res["chat_exp_5"] != "None":
        post_mental_models[policy]["general answer"].append(int(res["chat_exp_5"]))
    if res["chat_exp_6"] != "None":
        post_mental_models[policy]["personalized answer"].append(int(res["chat_exp_6"]))
    if res["chat_exp_7"] != "None":
        post_mental_models[policy]["immediate answer"].append(int(res['chat_exp_7']))
    if res["chat_exp_8"] != "None":
        post_mental_models[policy]["long dialog"].append(int(res["chat_exp_8"]))
    post_user_mms[user] = {key: post_mental_models[policy][key][-1] for key in post_mental_models[policy]}

### Compare Pre- and Post-Interaction mental models

In [111]:
for policy in ["cts", "faq", "hdc"]:
    print(f"POLICY: {policy}")
    for key in mental_models[policy]:
        print(f"CHANGE: {np.mean(post_mental_models[policy][key]) - np.mean(mental_models[policy][key])}")
        print({f"{key}: {stats.ttest_ind(mental_models[policy][key], post_mental_models[policy][key])}"})

POLICY: cts
CHANGE: 0.10000000000000009
{'natural language: Ttest_indResult(statistic=-0.30707133368359785, pvalue=0.7604656059019707)'}
CHANGE: 0.19999999999999973
{'keywords: Ttest_indResult(statistic=-0.5656854249492372, pvalue=0.5749329857222167)'}
CHANGE: -0.75
{'specific question: Ttest_indResult(statistic=2.707907232171109, pvalue=0.010092324480818786)'}
CHANGE: 0.34999999999999964
{'follow-up questions: Ttest_indResult(statistic=-1.1347643719320588, pvalue=0.2635815272695332)'}
CHANGE: 0.09999999999999964
{'general answer: Ttest_indResult(statistic=-0.2655201077955948, pvalue=0.7920440425247406)'}
CHANGE: 0.6499999999999999
{'personalized answer: Ttest_indResult(statistic=-2.035491539272913, pvalue=0.048818503499269)'}
CHANGE: -0.30000000000000027
{'immediate answer: Ttest_indResult(statistic=1.076718177564727, pvalue=0.28839461252596754)'}
CHANGE: 0.27894736842105283
{'long dialog: Ttest_indResult(statistic=-0.8288664065856189, pvalue=0.41249528960204906)'}
POLICY: faq
CHANGE:

**Takeaway:**
 * Mental models were largely not updated in the experimental group (system was able to adapt to match expected behavior), except in two instances where the mental models of users were exceeded (users felt they didn't need to phrase their question as preciesly as they had anticipated, but could still get a good result, and users felt that the chatbot was able to give them more personalised answers than they had expected)

## Was there any difference in change in mental models between groups?

In [100]:
change_dict = {"cts": {
                "natural language": [],
                "keywords": [],
                "specific question": [],
                "follow-up questions": [],
                "general answer": [],
                "personalized answer": [],
                "immediate answer": [],
                "long dialog": []},
              "faq": {      
                "natural language": [],
                "keywords": [],
                "specific question": [],
                "follow-up questions": [],
                "general answer": [],
                "personalized answer": [],
                "immediate answer": [],
                "long dialog": []}, 
              "hdc": {      
                "natural language": [],
                "keywords": [],
                "specific question": [],
                "follow-up questions": [],
                "general answer": [],
                "personalized answer": [],
                "immediate answer": [],
                "long dialog": []}}

for policy in mental_models:
    for key in mental_models[policy]:
        for i in range(len(mental_models[policy][key])):
            if i < len(post_mental_models[policy][key]):
              change_dict[policy][key].append(post_mental_models[policy][key][i] - mental_models[policy][key][i])

In [102]:
for key in change_dict["cts"]:
    print(key)
    print(stats.f_oneway(change_dict["cts"][key], change_dict["faq"][key], change_dict["hdc"][key]))

natural language
F_onewayResult(statistic=1.4578941018564604, pvalue=0.24098658490288474)
keywords
F_onewayResult(statistic=0.21352845507754686, pvalue=0.8083403086420449)
specific question
F_onewayResult(statistic=2.101749327771472, pvalue=0.1311525282334748)
follow-up questions
F_onewayResult(statistic=25.453837597330356, pvalue=1.260206774331302e-08)
general answer
F_onewayResult(statistic=2.4380380174515963, pvalue=0.0960852186026396)
personalized answer
F_onewayResult(statistic=4.182102781488613, pvalue=0.01993766847199302)
immediate answer
F_onewayResult(statistic=2.31711926903365, pvalue=0.10746182726259472)
long dialog
F_onewayResult(statistic=8.637968166272794, pvalue=0.0005287635759967562)


In [105]:
print("In general I think that a chatbot can ask clarifying questions to help me narrow down my problem, e.g., if my original question is vague")

print(np.mean(change_dict["cts"]["follow-up questions"]), np.mean(change_dict["faq"]["follow-up questions"]), np.mean(change_dict["hdc"]["follow-up questions"]))
print(stats.tukey_hsd(change_dict["cts"]["follow-up questions"], change_dict["faq"]["follow-up questions"], change_dict["hdc"]["follow-up questions"]))

In general I think that a chatbot can ask clarifying questions to help me narrow down my problem, e.g., if my original question is vague
0.35 -2.3333333333333335 -0.6363636363636364
Tukey's HSD Pairwise Group Comparisons (95.0% Confidence Interval)
Comparison  Statistic  p-value  Lower CI  Upper CI
 (0 - 1)      2.683     0.000     1.771     3.596
 (0 - 2)      0.986     0.022     0.119     1.854
 (1 - 0)     -2.683     0.000    -3.596    -1.771
 (1 - 2)     -1.697     0.000    -2.590    -0.804
 (2 - 0)     -0.986     0.022    -1.854    -0.119
 (2 - 1)      1.697     0.000     0.804     2.590



In [108]:
print("In general I think that a chatbot can give me a personalized answer specific to my case")

print(np.mean(change_dict["cts"]["personalized answer"]), np.mean(change_dict["faq"]["personalized answer"]), np.mean(change_dict["hdc"]["personalized answer"]))
print(stats.tukey_hsd(change_dict["cts"]["personalized answer"], change_dict["faq"]["personalized answer"], change_dict["hdc"]["personalized answer"]))

In general I think that a chatbot can give me a personalized answer specific to my case
0.65 -0.5714285714285714 -0.13636363636363635
Tukey's HSD Pairwise Group Comparisons (95.0% Confidence Interval)
Comparison  Statistic  p-value  Lower CI  Upper CI
 (0 - 1)      1.221     0.016     0.194     2.249
 (0 - 2)      0.786     0.159    -0.229     1.802
 (1 - 0)     -1.221     0.016    -2.249    -0.194
 (1 - 2)     -0.435     0.553    -1.438     0.568
 (2 - 0)     -0.786     0.159    -1.802     0.229
 (2 - 1)      0.435     0.553    -0.568     1.438



**Takeaways:**
 * Users updated their mental model, to expect more personalised information after interacting with the chatbot

In [109]:
print("In general I think that a chatbot would need to ask multiple questions before it is able to give me an answer")

print(np.mean(change_dict["cts"]["long dialog"]), np.mean(change_dict["faq"]["long dialog"]), np.mean(change_dict["hdc"]["long dialog"]))
print(stats.tukey_hsd(change_dict["cts"]["long dialog"], change_dict["faq"]["long dialog"], change_dict["hdc"]["long dialog"]))

In general I think that a chatbot would need to ask multiple questions before it is able to give me an answer
0.2631578947368421 -1.45 0.2857142857142857
Tukey's HSD Pairwise Group Comparisons (95.0% Confidence Interval)
Comparison  Statistic  p-value  Lower CI  Upper CI
 (0 - 1)      1.713     0.002     0.545     2.882
 (0 - 2)     -0.023     0.999    -1.177     1.132
 (1 - 0)     -1.713     0.002    -2.882    -0.545
 (1 - 2)     -1.736     0.002    -2.875    -0.596
 (2 - 0)      0.023     0.999    -1.132     1.177
 (2 - 1)      1.736     0.002     0.596     2.875

